In [24]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [25]:
import sqlite3

In [38]:
SOURCE_DATA_PATH = "C:/Users/liivas/Downloads/Töö/estnltk_projekt_2025/morphology_conflicts/data/"
SOURCE_DATA = "verb_obj_cases_strict.db"
SOURCE_DATA2 = "verb_obj_cases_all.db"
TR_SOURCE_DATA_PATH = "C:/Users/liivas/Downloads/Töö/estnltk_projekt_2025/transaktsioonid/source_data/"
TR_DB = "v33_koondkorpus_sentences_verb_pattern_obl_20241002-130310.db"
RESULT_ALL = "verb_obj_all_strict_examples.db"
RESULT_NOM = "verb_obj_nom_strict.db"
RESULT_GEN = "verb_obj_gen_strict.db"
RESULT_PART = "verb_obj_part_strict.db"
RESULT_NOM2 = "verb_obj_nom_all.db"
RESULT_GEN2 = "verb_obj_gen_all.db"
RESULT_PART2 = "verb_obj_part_all.db"
SENTENCES = "v33_koondkorpus_sentences_sentences_20250220-130121.db"

In [27]:
# meetod käände saamiseks feats väljalt
def get_case(feats) -> str:
    cases = ["nom", 
             "gen", 
             "part", 
             "adit", 
             "ill", 
             "el", 
             "all", 
             "term", 
             "abl",
             "kom",
             "ad",
             "es",
             "abes",
             "tr"]
    
    feats_split = feats.split(",")
    case = ""
    # peaks olema 1 kääne per feats, aga järjekord pole fikseeritud
    for idx, feat in enumerate(feats_split):
        if feat in cases:
            case = feat
            break
    return case

In [28]:
# verbi aeg (tense)
# pres -> olevik
# past -> minevik

def get_verb_tense(feats: str) -> str|bool:
    if "pres" in feats:
        return "pres"
    elif "past" in feats:
        return "past"
    else:
        return ""

In [29]:
# verbi kõneviis
# käskiv -> imper
# tingiv -> cond
# kindel -> indic
# kaudne -> quot
# möönev -> juss

# muud on partic (mittefiniitsed verbivormid)

def get_verb_mood(feats: str) -> str|bool:
    if "imper" in feats:
        return "imper"
    elif "cond" in feats:
        return "cond"
    elif "indic" in feats:
        return "indic"
    elif "quot" in feats:
        return "quot"
    elif "juss" in feats:
        return "juss"
    else:
        if "partic" in feats:
            return "partic"
        else:
            return ""

In [39]:
# all
con = sqlite3.connect(f"{SOURCE_DATA_PATH}{RESULT_ALL}")

con.create_function("get_verb_tense", 1, get_verb_tense)
con.create_function("get_case", 1, get_case)
con.create_function("get_verb_mood", 1, get_verb_mood)

cur = con.cursor()
cur.execute(f'ATTACH DATABASE "{SOURCE_DATA_PATH}{SOURCE_DATA}" AS source')
cur.execute(f'ATTACH DATABASE "{TR_SOURCE_DATA_PATH}{TR_DB}" AS tr')
cur.execute(f'ATTACH DATABASE "{TR_SOURCE_DATA_PATH}{SENTENCES}" AS sents')

cur.execute("""
DROP TABLE IF EXISTS verb_strict_examples
""")

cur.execute("""
CREATE TABLE verb_strict_examples
AS
SELECT
    obj_cases.verb AS verb,
    obj_cases.verb_compound AS verb_compound,
    obj_cases.verb_tense AS verb_tense,
    obj_cases.verb_mood AS verb_mood,
    tr_row.form AS form,
    tr_row.lemma AS lemma,
    tr_row.loc AS loc,
    get_case(tr_row.feats) AS current_case,
    sentences.id AS sentence_id,
    sentences.text AS sentence
FROM
    tr.transaction_row AS tr_row
INNER JOIN
    tr.transaction_head AS tr_head
    ON tr_row.head_id = tr_head.id
INNER JOIN
    source.verbs_obj_case_lemma_percentages AS obj_cases
    ON tr_head.verb = obj_cases.verb
    AND tr_head.verb_compound = obj_cases.verb_compound
    AND get_verb_tense(tr_head.feats) = obj_cases.verb_tense
    AND get_verb_mood(tr_head.feats) = obj_cases.verb_mood
INNER JOIN
    sents.sentences AS sentences
    ON sentences.id = tr_head.sentence_id
WHERE
    tr_row.deprel = "obj"
GROUP BY
    obj_cases.verb,
    obj_cases.verb_compound,
    obj_cases.verb_tense,
    obj_cases.verb_mood,
    current_case
""")

con.close()

In [41]:
# teeme näidise

EXAMPLE_RES = "verb_obj_case_examples.db"

con = sqlite3.connect(f"{SOURCE_DATA_PATH}{EXAMPLE_RES}")

cur = con.cursor()
cur.execute(f'ATTACH DATABASE "{SOURCE_DATA_PATH}{RESULT_ALL}" AS source')

cur.execute("""
DROP TABLE IF EXISTS shuffled_verbs
""")

cur.execute("""
CREATE TABLE shuffled_verbs AS
SELECT DISTINCT
    verb,
    verb_compound,
    verb_tense,
    verb_mood
FROM
    source.verb_strict_examples
ORDER BY RANDOM()
LIMIT 10
""")

con.close()

In [43]:
con = sqlite3.connect(f"{SOURCE_DATA_PATH}{EXAMPLE_RES}")

cur = con.cursor()
cur.execute(f'ATTACH DATABASE "{SOURCE_DATA_PATH}{RESULT_ALL}" AS source')

cur.execute("""
DROP TABLE IF EXISTS shuffled_verb_all_rows
""")

cur.execute("""
CREATE TABLE shuffled_verb_all_rows AS
SELECT
    npse.*
FROM
    source.verb_strict_examples AS npse
INNER JOIN
    shuffled_verbs AS rvc
    ON npse.verb = rvc.verb
    AND npse.verb_compound = rvc.verb_compound
    AND npse.verb_tense = rvc.verb_tense
    AND npse.verb_mood = rvc.verb_mood
GROUP BY
    npse.verb,
    npse.verb_compound,
    npse.verb_tense,
    npse.verb_mood,
    npse.current_case
""")

con.close()

### NOM

In [ ]:
# verbi kõneviis
# käskiv -> imper
# tingiv -> cond
# kindel -> indic
# kaudne -> quot
# muud -> partic

In [30]:
con = sqlite3.connect(f"{SOURCE_DATA_PATH}{SOURCE_DATA}")
cur = con.cursor()

cur.execute(f'ATTACH DATABASE "{SOURCE_DATA_PATH}{RESULT_NOM}" AS result')

cur.execute("""
DROP TABLE IF EXISTS result.nom_probable
""")

cur.execute("""
CREATE TABLE result.nom_probable
AS
SELECT
    *
FROM 
    verbs_obj_case_lemma_percentages
WHERE
    percent_nom >= 80 
AND
    total >= 10
ORDER BY
    percent_nom
""")

con.close()

In [31]:
# obj lemmade saamine

con = sqlite3.connect(f"{SOURCE_DATA_PATH}{RESULT_NOM}")

con.create_function("get_verb_tense", 1, get_verb_tense)
con.create_function("get_case", 1, get_case)
con.create_function("get_verb_mood", 1, get_verb_mood)

cur = con.cursor()
cur.execute(f'ATTACH DATABASE "{TR_SOURCE_DATA_PATH}{TR_DB}" AS tr')

cur.execute(
    """
    SELECT
        tr_tbl.lemma
    FROM
    (
        SELECT
            tr_row.lemma AS lemma,
            tr_head.verb AS verb,
            tr_head.verb_compound AS verb_compound,
            get_verb_tense(tr_head.feats) AS verb_tense,
            get_verb_mood(tr_head.feats) AS verb_mood
        FROM
        (
            SELECT
                head_id,
                lemma
            FROM
                tr.transaction_row
            WHERE
                deprel = "obj"
            ) as tr_row
            INNER JOIN
                tr.transaction_head as tr_head
            ON
                tr_row.head_id=tr_head.id
        ) AS tr_tbl
        INNER JOIN
            nom_probable
        ON
            tr_tbl.verb = nom_probable.verb
        AND
            tr_tbl.verb_compound = nom_probable.verb_compound
        AND
            tr_tbl.verb_tense = nom_probable.verb_tense
        AND
            tr_tbl.verb_mood = nom_probable.verb_mood
""")

res = cur.fetchall()

con.close()

In [32]:
# alternatiiv
con = sqlite3.connect(f"{SOURCE_DATA_PATH}{RESULT_NOM}")

con.create_function("get_verb_tense", 1, get_verb_tense)
con.create_function("get_case", 1, get_case)
con.create_function("get_verb_mood", 1, get_verb_mood)

cur = con.cursor()
cur.execute(f'ATTACH DATABASE "{TR_SOURCE_DATA_PATH}{TR_DB}" AS tr')
cur.execute(f'ATTACH DATABASE "{TR_SOURCE_DATA_PATH}{SENTENCES}" AS sents')

cur.execute("""
SELECT
    tr_row.lemma
FROM
    tr.transaction_row AS tr_row
INNER JOIN
    tr.transaction_head AS tr_head
    ON tr_row.head_id = tr_head.id
INNER JOIN
    nom_probable
    ON tr_head.verb = nom_probable.verb
    AND tr_head.verb_compound = nom_probable.verb_compound
    AND get_verb_tense(tr_head.feats) = nom_probable.verb_tense
    AND get_verb_mood(tr_head.feats) = nom_probable.verb_mood
WHERE
    tr_row.deprel = "obj"
""")

res = cur.fetchall()

con.close()

In [13]:
len(res)

10328

In [14]:
res[:5]

[('silm',), ('missugune',), ('hea',), ('päev',), ('noorem',)]

In [15]:
test_res = res

In [16]:
len(test_res)

10328

In [33]:
con = sqlite3.connect(f"{SOURCE_DATA_PATH}{RESULT_NOM}")

con.create_function("get_verb_tense", 1, get_verb_tense)
con.create_function("get_case", 1, get_case)
con.create_function("get_verb_mood", 1, get_verb_mood)

cur = con.cursor()
cur.execute(f'ATTACH DATABASE "{TR_SOURCE_DATA_PATH}{TR_DB}" AS tr')
cur.execute(f'ATTACH DATABASE "{TR_SOURCE_DATA_PATH}{SENTENCES}" AS sents')

cur.execute("""
DROP TABLE IF EXISTS nom_probable_strict_examples
""")

cur.execute("""
CREATE TABLE nom_probable_strict_examples
AS
SELECT
    nom_probable.verb AS verb,
    nom_probable.verb_compound AS verb_compound,
    nom_probable.verb_tense AS verb_tense,
    nom_probable.verb_mood AS verb_mood,
    tr_row.form AS form,
    tr_row.lemma AS lemma,
    tr_row.loc AS loc,
    get_case(tr_row.feats) AS current_case,
    sentences.id AS sentence_id,
    sentences.text AS sentence
FROM
    tr.transaction_row AS tr_row
INNER JOIN
    tr.transaction_head AS tr_head
    ON tr_row.head_id = tr_head.id
INNER JOIN
    nom_probable
    ON tr_head.verb = nom_probable.verb
    AND tr_head.verb_compound = nom_probable.verb_compound
    AND get_verb_tense(tr_head.feats) = nom_probable.verb_tense
    AND get_verb_mood(tr_head.feats) = nom_probable.verb_mood
INNER JOIN
    sents.sentences AS sentences
    ON sentences.id = tr_head.sentence_id
WHERE
    tr_row.deprel = "obj"
GROUP BY
    nom_probable.verb,
    nom_probable.verb_compound,
    nom_probable.verb_tense,
    nom_probable.verb_mood,
    current_case
""")

con.close()

In [17]:
import random

random.shuffle(test_res)

### GEN

In [34]:
con = sqlite3.connect(f"{SOURCE_DATA_PATH}{SOURCE_DATA}")
cur = con.cursor()

cur.execute(f'ATTACH DATABASE "{SOURCE_DATA_PATH}{RESULT_GEN}" AS result')

cur.execute("""
DROP TABLE IF EXISTS result.gen_probable
""")

cur.execute("""
CREATE TABLE result.gen_probable
AS
SELECT
    *
FROM 
    verbs_obj_case_lemma_percentages
WHERE
    percent_gen >= 80 
AND
    total >= 10
ORDER BY
    percent_gen
""")

con.close()

In [35]:
con = sqlite3.connect(f"{SOURCE_DATA_PATH}{RESULT_GEN}")

con.create_function("get_verb_tense", 1, get_verb_tense)
con.create_function("get_case", 1, get_case)
con.create_function("get_verb_mood", 1, get_verb_mood)

cur = con.cursor()
cur.execute(f'ATTACH DATABASE "{TR_SOURCE_DATA_PATH}{TR_DB}" AS tr')
cur.execute(f'ATTACH DATABASE "{TR_SOURCE_DATA_PATH}{SENTENCES}" AS sents')

cur.execute("""
DROP TABLE IF EXISTS gen_probable_strict_examples
""")

cur.execute("""
CREATE TABLE gen_probable_strict_examples
AS
SELECT
    gen_probable.verb AS verb,
    gen_probable.verb_compound AS verb_compound,
    gen_probable.verb_tense AS verb_tense,
    gen_probable.verb_mood AS verb_mood,
    tr_row.form AS form,
    tr_row.lemma AS lemma,
    tr_row.loc AS loc,
    get_case(tr_row.feats) AS current_case,
    sentences.id AS sentence_id,
    sentences.text AS sentence
FROM
    tr.transaction_row AS tr_row
INNER JOIN
    tr.transaction_head AS tr_head
    ON tr_row.head_id = tr_head.id
INNER JOIN
    gen_probable
    ON tr_head.verb = gen_probable.verb
    AND tr_head.verb_compound = gen_probable.verb_compound
    AND get_verb_tense(tr_head.feats) = gen_probable.verb_tense
    AND get_verb_mood(tr_head.feats) = gen_probable.verb_mood
INNER JOIN
    sents.sentences AS sentences
    ON sentences.id = tr_head.sentence_id
WHERE
    tr_row.deprel = "obj"
GROUP BY
    gen_probable.verb,
    gen_probable.verb_compound,
    gen_probable.verb_tense,
    gen_probable.verb_mood,
    current_case
""")

con.close()

### PART

In [36]:
con = sqlite3.connect(f"{SOURCE_DATA_PATH}{SOURCE_DATA}")
cur = con.cursor()

cur.execute(f'ATTACH DATABASE "{SOURCE_DATA_PATH}{RESULT_PART}" AS result')

cur.execute("""
DROP TABLE IF EXISTS result.part_probable
""")

cur.execute("""
CREATE TABLE result.part_probable
AS
SELECT
    *
FROM 
    verbs_obj_case_lemma_percentages
WHERE
    percent_par >= 80 
AND
    total >= 10
ORDER BY
    percent_par
""")

con.close()

In [37]:
con = sqlite3.connect(f"{SOURCE_DATA_PATH}{RESULT_PART}")

con.create_function("get_verb_tense", 1, get_verb_tense)
con.create_function("get_case", 1, get_case)
con.create_function("get_verb_mood", 1, get_verb_mood)

cur = con.cursor()
cur.execute(f'ATTACH DATABASE "{TR_SOURCE_DATA_PATH}{TR_DB}" AS tr')
cur.execute(f'ATTACH DATABASE "{TR_SOURCE_DATA_PATH}{SENTENCES}" AS sents')

cur.execute("""
DROP TABLE IF EXISTS part_probable_strict_examples
""")

cur.execute("""
CREATE TABLE part_probable_strict_examples
AS
SELECT
    part_probable.verb AS verb,
    part_probable.verb_compound AS verb_compound,
    part_probable.verb_tense AS verb_tense,
    part_probable.verb_mood AS verb_mood,
    tr_row.form AS form,
    tr_row.lemma AS lemma,
    tr_row.loc AS loc,
    get_case(tr_row.feats) AS current_case,
    sentences.id AS sentence_id,
    sentences.text AS sentence
FROM
    tr.transaction_row AS tr_row
INNER JOIN
    tr.transaction_head AS tr_head
    ON tr_row.head_id = tr_head.id
INNER JOIN
    part_probable
    ON tr_head.verb = part_probable.verb
    AND tr_head.verb_compound = part_probable.verb_compound
    AND get_verb_tense(tr_head.feats) = part_probable.verb_tense
    AND get_verb_mood(tr_head.feats) = part_probable.verb_mood
INNER JOIN
    sents.sentences AS sentences
    ON sentences.id = tr_head.sentence_id
WHERE
    tr_row.deprel = "obj"
GROUP BY
    part_probable.verb,
    part_probable.verb_compound,
    part_probable.verb_tense,
    part_probable.verb_mood,
    current_case
""")

con.close()

### Lenient (vormihomonüümia sees)

In [6]:
con = sqlite3.connect(f"{SOURCE_DATA_PATH}{SOURCE_DATA2}")
cur = con.cursor()

cur.execute(f'ATTACH DATABASE "{SOURCE_DATA_PATH}{RESULT_NOM2}" AS result')

cur.execute("""
DROP TABLE IF EXISTS result.nom_probable
""")

cur.execute("""
CREATE TABLE result.nom_probable
AS
SELECT
    *
FROM 
    verbs_obj_case_lemma_all_percentages
WHERE
    percent_nom >= 80 
AND
    total >= 10
ORDER BY
    percent_nom
""")

con.close()

In [7]:
con = sqlite3.connect(f"{SOURCE_DATA_PATH}{SOURCE_DATA2}")
cur = con.cursor()

cur.execute(f'ATTACH DATABASE "{SOURCE_DATA_PATH}{RESULT_GEN2}" AS result')

cur.execute("""
DROP TABLE IF EXISTS result.gen_probable
""")

cur.execute("""
CREATE TABLE result.gen_probable
AS
SELECT
    *
FROM 
    verbs_obj_case_lemma_all_percentages
WHERE
    percent_gen >= 80 
AND
    total >= 10
ORDER BY
    percent_gen
""")

con.close()

In [8]:
con = sqlite3.connect(f"{SOURCE_DATA_PATH}{SOURCE_DATA2}")
cur = con.cursor()

cur.execute(f'ATTACH DATABASE "{SOURCE_DATA_PATH}{RESULT_PART2}" AS result')

cur.execute("""
DROP TABLE IF EXISTS result.part_probable
""")

cur.execute("""
CREATE TABLE result.part_probable
AS
SELECT
    *
FROM 
    verbs_obj_case_lemma_all_percentages
WHERE
    percent_par >= 80 
AND
    total >= 10
ORDER BY
    percent_par
""")

con.close()